# Functions of Several Variables

Every function so far has taken one number and returned one number, and its graph was a curve you could draw on a page. Almost nothing worth predicting works that way. A patient's response to therapy depends on the dose of one drug *and* the dose of another; a glucose reading six months out depends on age *and* BMI *and* blood pressure. This notebook is the companion to the **Functions of Several Variables** and **Functions in Machine Learning** sections of the Unit 1-3 slides: one notebook section per slide topic, in the same order, with the pictures made interactive so you can rotate, slide and search them yourself.

## Learning objectives

- Evaluate a function of several variables, and say why the input is an **ordered** list.
- Read the graph of $f(x,y)$ as a **surface**, and rotate one in a live window.
- Read a **level curve** map, and connect crowded curves to a steep response.
- Take a **slice** through a surface and recover an ordinary one-variable function.
- Describe where the inputs live (a **Cartesian product**) and which of them are allowed (the **domain**).
- Read a data table as a set of tuples, and a **model** as a function from features to a label.
- Explain why the prediction **error** is itself a function, of the parameters rather than of the data.

## Background

You need only what a function is: a rule that turns each input into **exactly one** output, written $f(x)$, with a **domain** (the inputs the rule accepts) and a **range** (the outputs it actually produces). If a rule ever returned two different answers for the same input it would not be a function.

You should also recognise a few one-variable shapes by name, because they reappear here as slices of a surface: a **line** $mx + b$, and a **parabola** $ax^2 + bx + c$, which opens downward when $a$ is negative. Everything else is developed below.

## This notebook covers

1. Two inputs, one output
2. The letters are ours to choose
3. Where the inputs live: the Cartesian product
4. The graph of $f(x,y)$ is a surface (rotatable)
5. Level curves
6. Slices (with a slider)
7. Domain and range, with several inputs
8. A data table is a set of tuples
9. A model is a function
10. Error is a function too
11. Solving it: predicted versus true
12. Summary

**Prerequisites:** none are strictly required; `U1-3_Functions-1_DomainRangeAndFamilies` develops domain, range and the one-variable families at full length, and `U1-2_Sets-1_Operations` develops sets, which section 3 builds the Cartesian product from.

**Dataset:** none external. Every table here is six invented patients, typed out in section 8 so you can see exactly where every number comes from.

### Before you run anything

Most of this notebook draws ordinary flat pictures, and those are best left **inline**, printed in the notebook where you can scroll back to them. That is the default set below.

The two **3-D** pictures are the exception. A surface is worth rotating, and a static image cannot be rotated, so those two cells switch matplotlib to the `qt` backend first, which opens the figure in **a separate window you can drag with the mouse**. Two practical notes about those windows:

- The window sometimes opens **behind** your browser or editor. Check the taskbar if a plot seems not to appear.
- Switching back to inline plots **closes** any open `qt` window, so rotate the surface while you have it, then move on. The cell after each 3-D figure switches back for you.
- If `qt` is not installed on your machine, `interactive_3d()` says so and leaves you with an inline 3-D plot. Everything still runs; you just lose the dragging.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D     # noqa: F401  (registers the 3d projection)
from ipywidgets import interact, FloatSlider

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

plt.style.use('dark_background')

import warnings
warnings.filterwarnings('ignore')

# shared display helper (msds520_helpers.py lives at the MSDS 520 course root).
# notebooks sit two folders below it, so '../..' points back to it.
import sys
sys.path.append('../..')
import msds520_helpers as helpers


# plotting helper: draw x-y graphs like the usual coordinate plane
def center_axes(ax):
    ax.spines['left'].set_position('zero')     # y-axis through x = 0
    ax.spines['bottom'].set_position('zero')   # x-axis through y = 0
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)


# --- backend policy: inline everywhere, qt only for the two 3-D surfaces ---
def interactive_3d():
    """Send the next figure to a draggable qt window."""
    try:
        get_ipython().run_line_magic('matplotlib', 'qt')
    except Exception as err:
        print(f'qt backend unavailable ({err}); drawing inline instead, so no dragging.')


def static_plots():
    """Back to inline figures. NOTE: this closes any open qt window."""
    get_ipython().run_line_magic('matplotlib', 'inline')


static_plots()          # inline is the default for everything below

## 1. Two inputs, one output

A **function of $p$ variables** takes an ordered list of $p$ numbers and returns a single value:

$$f(x_1, x_2, \ldots, x_p) = \text{one output}.$$

Nothing about the definition of a function has changed. "Exactly one output per input" still holds; what changed is that a single input is now a whole list rather than a single number. Everything earlier in this lecture was the case $p = 1$.

Body mass index is the smallest useful example. It takes a weight $w$ in kilograms and a height $h$ in metres:

$$\mathrm{BMI}(w, h) = \frac{w}{h^{2}}.$$

Two inputs, one output. The input is a **pair**, and the pair is **ordered**: $\mathrm{BMI}(70, 1.8)$ and $\mathrm{BMI}(1.8, 70)$ are not the same question, and only one of them is about a person.

In [ ]:
def bmi(w, h):
    """weight in kg, height in m, one number out."""
    return w / h**2


print(f'BMI(70, 1.8) = {bmi(70, 1.8):.2f}')
print(f'BMI(1.8, 70) = {bmi(1.8, 70):.6f}   <- the same two numbers, swapped')

The second call is not a mistake the computer can catch: it is a perfectly good arithmetic question about a person weighing 1.8 kg who is 70 m tall. **Order is part of the input.** This is exactly why we will write inputs as ordered tuples $(w, h)$ rather than as an unordered set $\{w, h\}$.

A function of several variables is just as happy to take a whole column of inputs at once, which is how we will feed a whole grid of points to the surface plot in section 4.

In [ ]:
weights = np.array([55.0, 70.0, 85.0, 100.0])
heights = np.array([1.60, 1.70, 1.80, 1.90])

for w, h in zip(weights, heights):
    print(f'BMI({w:5.1f}, {h:.2f}) = {bmi(w, h):5.2f}')

## 2. The letters are ours to choose

These three lines all describe the same kind of object, **two inputs and one output**:

$$y = f(x_1, x_2), \qquad z = f(x, y), \qquad \mathrm{BMI} = f(w, h).$$

Two naming conventions are in common use, and both are standard.

- **Subscripts**, $f(x_1, x_2, \ldots, x_p)$, scale to any number of inputs. Once $p$ is large this is the only sane choice, and it is what data science uses.
- **Separate letters**, $f(x, y)$ or $f(w, h)$, are convenient when $p$ is 2 or 3, because each input gets its own name and its own axis in a picture.

**The trap.** In $y = f(x_1, x_2)$ the letter $y$ is the *output*, and it is plotted upward. In $z = f(x, y)$ that same letter $y$ is an *input*, plotted along a horizontal axis, and $z$ is the output. The letter $y$ does not mean "the vertical axis"; it means whatever the line of algebra says it means.

> **Position tells you the role, not the letter.** Whatever sits inside the parentheses is an input. Whatever $f$ equals is the output.

The code below writes one single rule three times, under three sets of names, and checks that all three give the same number for the same inputs.

In [ ]:
def as_subscripts(x1, x2):
    return x1 / x2**2

def as_xy(x, y):
    return x / y**2

def as_wh(w, h):
    return w / h**2


a, b = 70.0, 1.8
print(f'y = f(x1, x2) with x1={a}, x2={b}   ->  y = {as_subscripts(a, b):.4f}')
print(f'z = f(x,  y)  with x ={a}, y ={b}   ->  z = {as_xy(a, b):.4f}')
print(f'BMI = f(w, h) with w ={a}, h ={b}   ->  BMI = {as_wh(a, b):.4f}')
print()
print(f'all three agree: {as_subscripts(a, b) == as_xy(a, b) == as_wh(a, b)}')

Notice what the middle line does to your reading habits: `y` there is an **input**, the height, and it is the second thing in the parentheses. Nothing in the notation stops a letter from playing a different role in a different formula, so read the parentheses, not the alphabet.

## 3. Where the inputs live: the Cartesian product

The inputs of a two-variable function are ordered pairs, so before asking which pairs are *allowed* we need a name for the set of **all** pairs. Given two sets $A$ and $B$, their **Cartesian product** is

$$A \times B = \{\, (a, b) \;:\; a \in A, \; b \in B \,\},$$

every element of $A$ paired with every element of $B$, first coordinate from $A$ and second from $B$. For $p$ sets, $A_1 \times A_2 \times \cdots \times A_p$ is the set of ordered $p$-tuples built the same way.

Counting is easy: pick any of the $|A|$ first coordinates, then independently any of the $|B|$ second coordinates, so

$$|A \times B| = |A| \cdot |B|.$$

Taking $A = B = \mathbb{R}$ gives every pair of real numbers, which is the whole plane $\mathbb{R}^{2}$; $p$ copies of $\mathbb{R}$ give $\mathbb{R}^{p}$.

In [ ]:
from itertools import product

A = {1, 2}
B = {'x', 'y', 'z'}

pairs = sorted(product(sorted(A), sorted(B)))

print(f'A x B = {pairs}')
print(f'|A| = {len(A)}, |B| = {len(B)}, |A x B| = {len(pairs)}')
print(f'|A| * |B| = {len(A) * len(B)}')

With numbers instead of letters, a product of two intervals is a **rectangle** of points in the plane. The plot below draws $A \times B$ for $A = [1, 4]$ and $B = [1, 3]$ as a grid of sample pairs: the first coordinate ranges over $A$ along the horizontal axis, the second over $B$ along the vertical.

In [ ]:
grid_x = np.linspace(1, 4, 13)      # sampled from A = [1, 4]
grid_y = np.linspace(1, 3, 9)       # sampled from B = [1, 3]
GX, GY = np.meshgrid(grid_x, grid_y)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(GX, GY, s=10, color='tab:cyan')
ax.plot(2.6, 2.1, marker='o', color='red', markersize=8)
ax.annotate('(a, b)', (2.6, 2.1), textcoords='offset points', xytext=(8, 6), color='red')

ax.set_xlim(-0.6, 4.8)
ax.set_ylim(-0.6, 3.8)
center_axes(ax)
ax.set_xlabel('x1', loc='right')
ax.set_ylabel('x2', loc='top', rotation=0)
ax.set_title('A x B for A = [1, 4] and B = [1, 3]')

plt.tight_layout()
plt.show()

Note which letters went on the axes. The first coordinate of the pair is $x_1$ and the second is $x_2$; the names $A$ and $B$ describe *which set each coordinate is drawn from*, not the axes themselves. This is section 2's warning in picture form.

## 4. The graph of $f(x,y)$ is a surface

One input and one output gives a **curve**: for each $x$ on the horizontal axis, the height of the curve is $f(x)$. Two inputs and one output need one more axis. The two inputs spread out over a **floor** (the horizontal plane), and above each floor point the function's value is a **height**. Joining up all those heights gives a **surface**.

### The example: two drugs given together

Two drugs are given together. Let $x$ be the dose of drug A in mg, $y$ the dose of drug B in mg, and let $P(x,y)$ be the percent of symptoms relieved. Too little of either drug does nothing; too much brings side effects that undo the benefit. A simple model of that behaviour is

$$P(x,y) = 100 - 0.04\,(x - 40)^{2} - 0.09\,(y - 25)^{2}.$$

Read the formula before you look at the picture. A square is never negative, and both squares are **subtracted** from 100, so:

- the largest value $P$ can possibly take is $100$;
- it takes that value only when **both** squares are zero, that is at $x = 40$ and $y = 25$;
- any other dose pair subtracts something, so the surface falls away in every direction from that one point.

A surface shaped like that is a **dome**, and its top is the best combination of the two drugs.

In [ ]:
def percent_relief(x, y):
    """percent symptom relief for dose x of drug A and dose y of drug B."""
    return 100 - 0.04 * (x - 40)**2 - 0.09 * (y - 25)**2


# the floor: every dose pair we are willing to consider
xs = np.linspace(10, 70, 60)      # mg of drug A
ys = np.linspace(5, 45, 60)       # mg of drug B
X, Y = np.meshgrid(xs, ys)        # X[i,j], Y[i,j] is one floor point
Z = percent_relief(X, Y)                # the height above it

print(f'the floor is a grid of {X.size} dose pairs')
print(f'P at the peak      P(40, 25) = {percent_relief(40, 25):6.2f}')
print(f'P off the peak     P(40, 35) = {percent_relief(40, 35):6.2f}')
print(f'P off the peak     P(50, 25) = {percent_relief(50, 25):6.2f}')
print(f'largest P anywhere on the grid = {Z.max():6.2f}')

Now the surface itself. **Drag inside the window with the mouse to rotate it**, and look at the dome from several angles: from the side it looks like an arch, and from directly overhead it turns into the contour map of the next section.

In [ ]:
interactive_3d()          # this one is worth rotating

fig = plt.figure(figsize=(9, 6.5))
ax = fig.add_subplot(projection='3d')

ax.plot_surface(X, Y, Z, cmap='viridis', edgecolor='none', alpha=0.9)
ax.scatter([40], [25], [percent_relief(40, 25)], color='red', s=45)   # the peak

ax.set_xlabel('x: dose of drug A (mg)')
ax.set_ylabel('y: dose of drug B (mg)')
ax.set_zlabel('P: percent relief')
ax.set_title('P(x, y) = 100 - 0.04(x-40)^2 - 0.09(y-25)^2   (drag to rotate)')

plt.tight_layout()
plt.show()

Two things to take away from the picture, both of which you could have predicted from the formula:

- Every floor point has **exactly one** height above it. That is the vertical line test, one dimension up: a surface that doubled back over itself would not be the graph of a function.
- The dome is **not** symmetric in the two directions. The coefficient on the $y$ term, $0.09$, is larger than the $0.04$ on the $x$ term, so the same overshoot in $y$ costs more relief than the same overshoot in $x$. Rotate until you are looking along each axis in turn and you can see the difference in steepness.

In [ ]:
static_plots()            # closes the 3-D window, back to inline figures

overshoot = 10        # mg too much of one drug

cost_x = percent_relief(40, 25) - percent_relief(40 + overshoot, 25)
cost_y = percent_relief(40, 25) - percent_relief(40, 25 + overshoot)

print(f'{overshoot} mg too much of drug A costs {cost_x:.1f} points of relief')
print(f'{overshoot} mg too much of drug B costs {cost_y:.1f} points of relief')
print(f'ratio = {cost_y / cost_x:.2f}, which is exactly 0.09 / 0.04')

## 5. Level curves

A **level curve** (or contour) collects every input pair that produces the *same* output:

$$\{\, (x, y) \;:\; P(x, y) = c \,\}.$$

This is the surface seen from directly overhead, and it is exactly how a topographic map shows a mountain: each line joins points of equal elevation, and the summit sits inside the innermost ring.

For our dome, setting $P(x,y) = c$ gives

$$0.04\,(x-40)^{2} + 0.09\,(y-25)^{2} = 100 - c,$$

which is the equation of an **ellipse** centred at $(40, 25)$. As $c$ climbs toward 100 the right-hand side shrinks toward zero and the ellipse shrinks toward a single point: the maximum.

Two habits worth building when reading any contour map:

- **crowded** curves mean the output is changing fast, a steep climb;
- **widely spaced** curves mean a flat plateau, where a small error in the input costs almost nothing.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))

levels = [40, 55, 70, 80, 90, 95, 99]
cs = ax.contour(X, Y, Z, levels=levels, cmap='viridis')
ax.clabel(cs, inline=True, fontsize=8, fmt='%.0f')
ax.plot(40, 25, marker='*', color='red', markersize=16)

ax.set_xlabel('x: dose of drug A (mg)')
ax.set_ylabel('y: dose of drug B (mg)')
ax.set_title('level curves of P: each ring is one value of c')

plt.tight_layout()
plt.show()

The rings are wider left-to-right than they are top-to-bottom, which is the same fact as before seen from above: it takes a bigger change in the dose of drug A than in the dose of drug B to lose the same amount of relief.

### 5.1 Finding the peak by search

We read the peak off the picture. A computer can find it without a picture, by evaluating $P$ on the whole grid and asking which floor point gave the largest height. That is the crudest possible **optimization**: try everything, keep the best. Unit 3 replaces it with calculus, which finds the same point without evaluating thousands of candidates.

In [ ]:
flat = np.argmax(Z)                        # index of the largest height
row, col = np.unravel_index(flat, Z.shape)
best_x, best_y, best_P = X[row, col], Y[row, col], Z[row, col]

print(f'best dose pair found on the grid: x = {best_x:.2f} mg, y = {best_y:.2f} mg')
print(f'relief there:                     P = {best_P:.3f}')
print(f'the formula says the true peak is at x = 40, y = 25 with P = {percent_relief(40, 25):.3f}')
print()
print(f'grid spacing in x: {xs[1] - xs[0]:.3f} mg,  in y: {ys[1] - ys[0]:.3f} mg')

The search lands close to the true peak but not exactly on it, and the reason is visible in the last line: the grid only contains the doses we happened to sample. A finer grid gets closer at the cost of more evaluations, which is precisely the trade that makes brute-force search a bad plan once there are more than two or three inputs.

## 6. Slices

Fix one of the two variables at a number, and the function collapses to an ordinary one-variable function of the other. Geometrically this is a **vertical cut** through the surface, and the curve exposed by the cut is a one-variable graph of the kind you already know how to read.

Clinically it is the everyday question: *I have fixed one drug, how much of the other should I give?*

Holding drug B at its best value $y = 25$ leaves

$$P(x, 25) = 100 - 0.04\,(x - 40)^{2},$$

a downward parabola with its vertex at $x = 40$. Holding drug A at $x = 40$ leaves $P(40, y) = 100 - 0.09\,(y - 25)^{2}$, a downward parabola with its vertex at $y = 25$. Both peak at 100, because both cuts pass through the top of the dome.

The interesting case is a cut that **misses** the top. The slider below moves the fixed dose of drug B. Drag it and watch two things: the parabola keeps its shape but slides **down**, and its peak stays at $x = 40$. Fixing the wrong dose of one drug does not change the best dose of the other here; it only costs you relief. (That convenient behaviour is special to this formula, and the ellipses of the previous section, whose axes line up with the coordinate axes, are the reason.)

In [ ]:
x_line = np.linspace(10, 70, 300)


@interact(y_fixed=FloatSlider(min=5.0, max=45.0, step=0.5, value=25.0,
                              description='dose of B (mg)', continuous_update=False))
def draw_slice(y_fixed):
    fig, ax = plt.subplots(figsize=(8, 4.6))

    ax.plot(x_line, percent_relief(x_line, y_fixed), lw=2, color='tab:cyan')
    ax.plot(40, percent_relief(40, y_fixed), marker='o', color='red')
    ax.axhline(100, ls=':', color='gray')            # the top of the whole dome

    ax.set_xlim(10, 70)
    ax.set_ylim(0, 108)
    ax.set_xlabel('x: dose of drug A (mg)')
    ax.set_ylabel('P: percent relief')
    ax.set_title(f'the slice at y = {y_fixed:.1f} mg peaks at P = {percent_relief(40, y_fixed):.1f}')

    plt.tight_layout()
    plt.show()

The dotted line is $P = 100$, the top of the whole dome. Only the slice at $y = 25$ reaches it; every other slice peaks below, and the shortfall is exactly the $0.09\,(y-25)^{2}$ that the other variable is costing you.

The two slices through the summit, one in each direction, are worth seeing side by side.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].plot(x_line, percent_relief(x_line, 25), lw=2, color='tab:cyan')
axes[0].plot(40, percent_relief(40, 25), marker='o', color='red')
axes[0].set_xlabel('x: dose of drug A (mg)')
axes[0].set_title('y = 25 fixed: P = 100 - 0.04(x-40)^2')

y_line = np.linspace(5, 45, 300)
axes[1].plot(y_line, percent_relief(40, y_line), lw=2, color='tab:orange')
axes[1].plot(25, percent_relief(40, 25), marker='o', color='red')
axes[1].set_xlabel('y: dose of drug B (mg)')
axes[1].set_title('x = 40 fixed: P = 100 - 0.09(y-25)^2')

for ax in axes:
    ax.set_ylim(0, 105)
    ax.set_ylabel('P: percent relief')

plt.tight_layout()
plt.show()

Both slices are parabolas opening downward, and the right-hand one is **narrower**: its coefficient $0.09$ is the larger of the two, so relief falls away faster as the dose of drug B drifts from 25. The width of a slice and the width of a level curve are two views of the same coefficient.

## 7. Domain and range, with several inputs

Nothing about domain and range changes when there are several inputs, except that the domain is now a set of **tuples** carved out of a Cartesian product:

$$f : D \to R, \qquad D \subseteq A_1 \times A_2 \times \cdots \times A_p.$$

A one-variable domain was an interval or a union of intervals on a line. A two-variable domain is a **region** in the plane, and it need not be a rectangle.

Two examples worth holding side by side.

- $\mathrm{BMI}(w, h) = w / h^{2}$: weight and height are both positive, and $h$ sits in a denominator, so $D = (0, \infty) \times (0, \infty)$, the open first quadrant. Every such pair gives a positive value, so $R = (0, \infty)$. This domain **is** a product: the restriction on $w$ and the restriction on $h$ are independent.
- $g(x, y) = \sqrt{9 - x^{2} - y^{2}}$: the square root forces $9 - x^{2} - y^{2} \geq 0$, that is $x^{2} + y^{2} \leq 9$. So $D$ is the **disk** of radius 3, which is *not* a product of two intervals: which values of $y$ are allowed depends on the $x$ you picked. Outputs run from 0 on the rim up to 3 at the centre, so $R = [0, 3]$.

In [ ]:
def g(x, y):
    """defined only where 9 - x^2 - y^2 >= 0."""
    return np.sqrt(9 - x**2 - y**2)


gx = np.linspace(-4, 4, 400)
gy = np.linspace(-4, 4, 400)
GXX, GYY = np.meshgrid(gx, gy)
allowed = (GXX**2 + GYY**2) <= 9            # True exactly on the domain

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.contourf(GXX, GYY, allowed.astype(float), levels=[0.5, 1.5], colors=['tab:cyan'], alpha=0.35)

ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
center_axes(ax)
ax.set_xlabel('x', loc='right')
ax.set_ylabel('y', loc='top', rotation=0)
ax.set_title('the domain of sqrt(9 - x^2 - y^2): a disk, not a rectangle')

plt.tight_layout()
plt.show()

inside = allowed.sum() / allowed.size
print(f'fraction of the plotted square that is inside the domain: {inside:.3f}')
print(f'g(0, 0) = {g(0, 0):.3f}   (the centre, the largest output)')
print(f'g(3, 0) = {g(3, 0):.3f}   (on the rim, the smallest output)')

The check `9 - x**2 - y**2 >= 0` is the whole domain question in one line of code. Ask it *before* calling the function: outside the disk the square root of a negative number is not a real value, and NumPy will hand back `nan` rather than raise, which is an easy way to poison a calculation without noticing.

In [ ]:
with np.errstate(invalid='ignore'):
    outside = g(3.0, 3.0)          # 9 - 9 - 9 = -9, outside the domain

print(f'g(3, 3) = {outside}   <- not a real number, and no error was raised')
print(f'is it nan? {np.isnan(outside)}')

## 8. A data table is a set of tuples

### 8.1 First, two words: features and label

Machine learning splits the columns of a data set into two roles, and every later idea depends on that split.

- A **feature** is one measured quantity you are allowed to use as an **input**: age, BMI, a lab value, a pixel, a word count. The features are written $x_1, x_2, \ldots, x_p$, so $p$ counts them.
- The **label** is the one quantity you want the model to **output**, written $y$. Learning from examples that already carry their labels is called **supervised** learning.

Three things about that split are worth saying out loud before looking at any data.

- **It is a decision, not a property of the data.** The same fasting glucose column is the label in this study, and could be a feature in a different one predicting, say, time to a first cardiac event.
- **A feature has to be available at the moment you predict.** A measurement taken *after* the answer is known cannot be an input to predicting it. Using one is a mistake serious enough to have its own name, a **leak**, and it produces models that look excellent and are worthless.
- **Labels are usually the expensive half.** Features are often already sitting in the chart, while the label may mean waiting six months, ordering a biopsy, or reading records by hand. That asymmetry is why real data sets tend to hold many unlabelled rows and few labelled ones.

### 8.2 The table

Six patients, three **features** each, and one **label**, their fasting glucose six months from now.

In [ ]:
patients = pd.DataFrame(
    {
        'age':      [54, 61, 47, 58, 43, 66],
        'bmi':      [27.4, 22.8, 31.5, 24.1, 33.2, 28.6],
        'systolic': [138, 122, 145, 118, 150, 131],
        'glucose':  [110, 101, 117, 102, 116, 116],
    },
    index=['A', 'B', 'C', 'D', 'E', 'F'],
)
patients

The first three columns are the features and the last is the label, so splitting the table is the first thing any modelling code does.

In [ ]:
FEATURE_COLS = ['age', 'bmi', 'systolic']
LABEL_COL = 'glucose'

features = patients[FEATURE_COLS]
labels = patients[LABEL_COL]

print(f'features (the inputs x): {FEATURE_COLS},  so p = {len(FEATURE_COLS)}')
print(f'label    (the output y): {LABEL_COL!r}')
print(f'patients (the rows):     n = {len(patients)}')

Now read that split three ways, because all three are saying the same thing in different languages.

- Each **feature column** is one of the sets being multiplied: the values that feature may take. Age lives in some interval of years, BMI in some interval of kg/m$^2$, systolic pressure in some interval of mmHg.
- Each **row** is one ordered tuple, one element of $A_1 \times A_2 \times A_3$, and one input to $f$. Patient A is the triple $(54, 27.4, 138)$.
- The **label column** holds the output we want the function to produce for that row.

So the domain of a real model is a Cartesian product, and your data table is a handful of points sampled from it. Unit 2 gives these tuples a name, **vectors**, and an algebra for handling thousands of them at once.

In [ ]:
row_A = features.loc['A'].to_numpy()

print(f'patient A as a tuple: {tuple(row_A)}')
print(f'that row is one point in R^{features.shape[1]}')
print(f'the table holds {features.shape[0]} such points')
print()
print('patient A as a column, the way Unit 2 will write it:')
helpers.show(row_A, name='x_1')

## 9. A model is a function

### 9.1 What machine learning is

In ordinary programming **you** write the rule and the computer applies it. That works whenever you can state the rule. Nobody can state the rule that turns a patient's chart into next year's glucose.

> **Machine learning is choosing a function from data.** You supply a *family* of candidate rules and a pile of solved examples; the computer picks the member of that family which best reproduces those examples, and you then use it on cases nobody has solved.

The thing being chosen is a **function**, and it maps the features to the label:

$$\underbrace{y}_{\text{label}} \;=\; \underbrace{f}_{\text{model}}\big( \underbrace{x_1, x_2, \ldots, x_p}_{\text{features}} \big), \qquad\text{that is,}\qquad \text{label} = \text{model}(\text{features}).$$

So everything earlier in this notebook applies directly: $f$ has a domain, a range, a family it is drawn from, and a graph you could in principle plot. Three ingredients are needed to run the procedure at all: **examples** to learn from, a **family** to choose within, and a measure of **how wrong** any particular choice is. Sections 10 and 11 build the last two.

One warning about the word. "Learning" here is not a metaphor for understanding: it is minimization, turning the parameters until the wrongness bottoms out. Nothing in the procedure knows what glucose is.

### 9.2 One row, one input, one output

Read one row of the table as a single input and a single output:

$$f(\underbrace{54}_{\text{age}},\; \underbrace{27.4}_{\text{BMI}},\; \underbrace{138}_{\text{BP}}) \;\approx\; \underbrace{110}_{\text{glucose}}.$$

Three inputs, one output, so a model is a function of $p = 3$ variables, exactly the subject of the first half of this notebook.

- Its **domain** is the set of measurement triples that make sense: the ages, BMIs and pressures a real patient could have, a region inside $\mathbb{R}^{3}$.
- Its **range** says what kind of answer comes out. A number, as here, makes it **regression**; a label from a finite set such as $\{\text{diabetic}, \text{not diabetic}\}$ would make it **classification**.

### 9.3 Naming the rows and columns

Six rows is small enough to point at. A real table has thousands, so the entries need names, and each name carries **two** indices: which row, and which column.

| | feature 1 | feature 2 | feature 3 | label |
|---|---|---|---|---|
| patient 1 | $x_{11}$ | $x_{12}$ | $x_{13}$ | $y_1$ |
| patient 2 | $x_{21}$ | $x_{22}$ | $x_{23}$ | $y_2$ |
| patient $n$ | $x_{n1}$ | $x_{n2}$ | $x_{n3}$ | $y_n$ |

- $x_{ij}$ is one **cell**: row $i$, column $j$.
- A whole **row** is the tuple $x_i = (x_{i1}, x_{i2}, \ldots, x_{ip})$, one input to $f$.
- $y_i$ is that row's label, so $n$ rows give the pairs $(x_1, y_1), \ldots, (x_n, y_n)$, the **training data**.

Read $(x_i, y_i)$ out loud as "patient $i$'s measurements, and patient $i$'s answer". The goal is to find one rule with $f(x_i) \approx y_i$ for every $i$, and, far more importantly, $f(x) \approx y$ for patients we have never measured.

### 9.4 Inputs versus parameters

Look again at the linear function with its two roles labelled:

$$f(x) = \underbrace{m}_{\text{parameter}} \, \underbrace{x}_{\text{input}} + \underbrace{b}_{\text{parameter}}.$$

The **input** $x$ changes from patient to patient. The **parameters** $m$ and $b$ are fixed once the model is trained; they say *which* line, out of the whole family of lines, we are using. With $p$ features the linear model just grows one term per feature,

$$f(x_1, \ldots, x_p) = w_1 x_1 + w_2 x_2 + \cdots + w_p x_p + b,$$

so there are $p + 1$ parameters to choose instead of two. Nothing conceptual changed; only the bookkeeping grew.

In [ ]:
def linear_model(x_row, weights, intercept):
    """one row of features in, one predicted label out."""
    return float(np.dot(weights, x_row) + intercept)


guess_w = np.array([0.5, 1.5, 0.2])      # a guess, not a fit: age, bmi, systolic
guess_b = 10.0

for name in patients.index:
    row = features.loc[name].to_numpy()
    pred = linear_model(row, guess_w, guess_b)
    print(f'patient {name}: predicted {pred:6.2f}, actual {labels[name]:3d}, '
          f'miss {pred - labels[name]:+6.2f}')

That guess is not terrible, which is a warning rather than a success: with three features there are many parameter choices that look plausible. To choose among them we need a single number that says how good a whole set of parameters is, and that number is the subject of the next section.

## 10. Error is a function too

To picture the error we need it to have exactly two inputs, so this section uses a single feature, BMI, and the one-feature linear model. Write the model's answer for row $i$ as $\hat{y}_i$, read "$y$ hat", to keep it apart from the truth $y_i$.

Once the data is fixed, the quality of a fit depends only on the parameters, so the error is a function of *those*, not of $x$:

$$
\begin{aligned}
\hat{y}_i \;&=\; m x_i + b
  &&\text{the model's prediction for row } i \\[4pt]
E(m, b) \;&=\; \sum_{i=1}^{n} \big( \underbrace{y_i}_{\text{truth}} - \underbrace{\hat{y}_i}_{\text{prediction}} \big)^{2}
  &&\text{total squared miss} \\[4pt]
\;&=\; \sum_{i=1}^{n} \big( y_i - (m x_i + b) \big)^{2}
  &&\text{the same thing, written out in full}
\end{aligned}
$$

The last line is the one to stare at. After the data is fixed every $x_i$ and $y_i$ is a **fixed number**, so the only symbols still free to vary are $m$ and $b$: the error really is a function of two variables, and everything in sections 4 to 6 applies to it. It has a surface, it has level curves, and it has a lowest point.

Squaring makes every miss count as positive, so an overshoot and an undershoot cannot cancel each other out. **Training** means finding the $m$ and $b$ that make $E$ as small as possible, and "find the input that minimizes a function" is **optimization**, the subject of Unit 3.

Notice the reversal, which is the central move in all of model fitting: the same symbols that were *unknowns to be estimated* a moment ago are the *inputs* of this new function.

In [ ]:
bmi_col = patients['bmi'].to_numpy()
glucose = patients['glucose'].to_numpy()


def sse(m, b):
    """sum of squared errors for the one-feature model m*bmi + b."""
    predictions = m * bmi_col + b
    return np.sum((glucose - predictions)**2)


print(f'E(1.0, 80.0) = {sse(1.0, 80.0):8.2f}')
print(f'E(1.5, 70.0) = {sse(1.5, 70.0):8.2f}')
print(f'E(0.0, {glucose.mean():.1f}) = {sse(0.0, glucose.mean()):8.2f}   '
      f'<- the flat model that always predicts the average')

In [ ]:
interactive_3d()          # rotate the bowl before running the next cell

m_vals = np.linspace(-0.5, 3.0, 160)
b_vals = np.linspace(30.0, 130.0, 160)
M, B = np.meshgrid(m_vals, b_vals)

E = np.array([[sse(m, b) for m in m_vals] for b in b_vals])

fig = plt.figure(figsize=(9, 6.5))
ax = fig.add_subplot(projection='3d')
ax.plot_surface(M, B, E, cmap='magma', edgecolor='none', alpha=0.9)
ax.set_xlabel('m (slope)')
ax.set_ylabel('b (intercept)')
ax.set_zlabel('E(m, b)')
ax.set_title('the error surface: a bowl over the parameters (drag to rotate)')

plt.tight_layout()
plt.show()

It is a **bowl**, and it opens upward because the squares are added rather than subtracted, the dome of section 3 turned upside down. Its lowest point is the best pair $(m, b)$.

The contour map makes the shape easier to read, and shows something the 3-D view hides: the level curves are long, thin, tilted ellipses. A long thin valley means many quite different parameter pairs give almost the same error, which is why fitted coefficients can be unstable when features are on awkward scales.

In [ ]:
static_plots()            # closes the 3-D window, back to inline figures

grid_min = np.unravel_index(np.argmin(E), E.shape)
m_hat_grid, b_hat_grid = M[grid_min], B[grid_min]

fig, ax = plt.subplots(figsize=(7.5, 5.5))
cs = ax.contour(M, B, E, levels=np.logspace(np.log10(E.min() + 1), np.log10(E.max()), 14),
                cmap='magma')
ax.clabel(cs, inline=True, fontsize=7, fmt='%.0f')
ax.plot(m_hat_grid, b_hat_grid, marker='*', color='red', markersize=16)

ax.set_xlabel('m (slope)')
ax.set_ylabel('b (intercept)')
ax.set_title('level curves of E, with the grid search minimum starred')

plt.tight_layout()
plt.show()

print(f'grid search minimum at m = {m_hat_grid:.4f}, b = {b_hat_grid:.4f}')
print(f'error there:            E = {E[grid_min]:.4f}')

The star is where our grid search bottomed out. Least squares can be solved exactly, without any searching, so we can check the answer.

In [ ]:
design = np.column_stack([bmi_col, np.ones_like(bmi_col)])       # [bmi | 1]
(m_hat, b_hat), *_ = np.linalg.lstsq(design, glucose, rcond=None)

print(f'exact least squares:  m = {m_hat:.4f}, b = {b_hat:.4f}, E = {sse(m_hat, b_hat):.4f}')
print(f'grid search:          m = {m_hat_grid:.4f}, b = {b_hat_grid:.4f}, E = {E[grid_min]:.4f}')
print()
print(f'the grid was spaced {m_vals[1] - m_vals[0]:.4f} apart in m '
      f'and {b_vals[1] - b_vals[0]:.4f} apart in b')

## 11. Solving it: predicted versus true

Now the full model, back to all three features:

$$\hat{y} = w_1 x_1 + w_2 x_2 + w_3 x_3 + b,$$

four parameters chosen to minimize the same sum of squares. There is no picture of $E$ this time, because it is a function of four variables and we have run out of axes, but the idea is identical: one number measuring how wrong a whole parameter set is, minimized.

In [ ]:
X_design = np.column_stack([features.to_numpy(), np.ones(len(features))])
coefficients, *_ = np.linalg.lstsq(X_design, labels.to_numpy(), rcond=None)

w_fit, b_fit = coefficients[:3], coefficients[3]

for name, w in zip(features.columns, w_fit):
    print(f'weight on {name:9s} = {w: .3f}')
print(f'intercept          = {b_fit: .3f}')

Each weight says how much the prediction moves when that feature goes up by one unit and the others stay put, which is exactly a **slice** in the sense of section 5. The units matter: a one-year change in age and a one-unit change in BMI are not comparable amounts of change, so the sizes of these numbers are not a ranking of importance.

With the parameters chosen, run every patient back through the rule and compare the answer to the truth.

In [ ]:
predicted = X_design @ coefficients
residuals = labels.to_numpy() - predicted

comparison = pd.DataFrame(
    {'true': labels.to_numpy(), 'predicted': predicted.round(2), 'residual': residuals.round(2)},
    index=patients.index,
)
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 5.6))

lo = min(labels.min(), predicted.min()) - 2
hi = max(labels.max(), predicted.max()) + 2

ax.plot([lo, hi], [lo, hi], ls='--', color='gray')          # perfect prediction
ax.scatter(labels, predicted, s=45, color='tab:cyan', zorder=3)
for name in patients.index:
    ax.annotate(name, (labels[name], comparison.loc[name, 'predicted']),
                textcoords='offset points', xytext=(6, 4), fontsize=9)

ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_aspect('equal')
ax.set_xlabel('true glucose y_i (mg/dL)')
ax.set_ylabel('predicted f(x_i) (mg/dL)')
ax.set_title('predicted vs. true: the dashed line is perfect prediction')

plt.tight_layout()
plt.show()

Each dot is one patient, with the true label across and the model's answer up. The dashed line is $\hat{y}_i = y_i$, perfect prediction, so a dot's vertical distance from that line is that patient's error. Dots above the line are patients the model **over**-predicted; dots below it were under-predicted.

Three standard summaries of those distances:

$$\mathrm{SSE} = \sum_i \big(y_i - \hat{y}_i\big)^{2}, \qquad
\mathrm{RMSE} = \sqrt{\mathrm{SSE}/n}, \qquad
\mathrm{MAE} = \frac{1}{n}\sum_i \big| y_i - \hat{y}_i \big|.$$

SSE is the quantity that was minimized, but it is hard to interpret because it is in squared mg/dL and it grows with the number of patients. RMSE fixes both problems: dividing by $n$ removes the dependence on sample size, and the square root puts the answer back **in the label's own units**, so it can be compared directly against the glucose values themselves. MAE is the plain average miss, less sensitive to a single large error.

In [ ]:
n = len(labels)
sse_full = float(np.sum(residuals**2))
rmse = float(np.sqrt(sse_full / n))
mae = float(np.mean(np.abs(residuals)))

print(f'SSE  = {sse_full:7.2f}  (squared mg/dL)')
print(f'RMSE = {rmse:7.2f}  mg/dL')
print(f'MAE  = {mae:7.2f}  mg/dL')
print()
print(f'for scale, the glucose values run from {labels.min()} to {labels.max()} mg/dL, '
      f'mean {labels.mean():.1f}')
print(f'always predicting that mean would give RMSE = '
      f'{np.sqrt(np.mean((labels - labels.mean())**2)):.2f} mg/dL')

### 11.1 An honest warning

The last comparison is the one to take seriously: the fitted rule beats the do-nothing rule that always predicts the average. But there are **six rows and four parameters** here, which is nearly an even match, and a flexible rule can always be bent to pass close to a few points. An error measured on the same patients the parameters were chosen from will flatter the model.

The fix is to measure error on patients the fit never saw, which is what a train/test split is for. That is MSDS 565's subject, and it is worth knowing now that the number printed above is an optimistic one.

In [ ]:
# leave one patient out, refit on the other five, predict the one left out
held_out_errors = []

for name in patients.index:
    keep = patients.index != name
    Xk = np.column_stack([features[keep].to_numpy(), np.ones(keep.sum())])
    coef_k, *_ = np.linalg.lstsq(Xk, labels[keep].to_numpy(), rcond=None)

    x_new = np.append(features.loc[name].to_numpy(), 1.0)
    held_out_errors.append(labels[name] - float(x_new @ coef_k))

held_out_errors = np.array(held_out_errors)

print(f'RMSE on the rows used for fitting: {rmse:6.2f} mg/dL')
print(f'RMSE on held-out patients:         {np.sqrt(np.mean(held_out_errors**2)):6.2f} mg/dL')

The second number is the honest one, and on six patients it is dramatically worse. That gap is the whole reason machine learning insists on evaluating a model on data it has never seen.

## 12. Summary

- A **function of several variables** takes an ordered tuple of $p$ numbers and returns exactly one output. Order is part of the input.
- The **letters are ours to choose**: $y = f(x_1, x_2)$ and $z = f(x, y)$ say the same thing. Position in the notation tells you whether a symbol is an input or the output; the letter does not.
- The graph of $f(x, y)$ is a **surface**: the inputs spread over a floor, and the output is the height above each floor point. Exactly one height per floor point is the vertical line test, one dimension up.
- A **level curve** joins all the input pairs sharing one output value, the surface seen from above. Crowded curves mean a steep response, widely spaced curves a flat plateau, and curves closing in on a point mean a maximum or a minimum there.
- A **slice** fixes one input and leaves an ordinary one-variable function, which is the practical question "I have fixed one thing, how much of the other?"
- The inputs live in a **Cartesian product** $A_1 \times \cdots \times A_p$, and the **domain** is the subset of it the formula accepts, a region in the plane rather than an interval on a line.
- A **data table** is a set of tuples: one row is one input, one column is one feature, and the label column is what the function should produce.
- A **model** is a function from features to a label, and its **error** is a function of the parameters. Minimizing that error is training, and "find the input that minimizes a function" is optimization, which is Unit 3.
- Error measured on the rows used for fitting is optimistic. Held-out data is the honest measure.

Next: adding up long or endless lists of numbers, **series and convergence** (`U1-4_Series-1_Convergence`).